# Multi-Model Output Comparison

Compare energy, CO2, and temperature metrics across multiple EnergyPlus output files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ── Column mapping ──────────────────────────────────────────────────────────

COLUMN_NAMES = [
    'Time',
    'Outdoor_Tdb_C',
    'Outdoor_Twb_C',
    'Space1_occupants', 'Space2_occupants', 'Space3_occupants',
    'Space4_occupants', 'Space5_occupants',
    'lights-1', 'lights-2', 'lights-3', 'lights-4', 'lights-5',
    'equip-1', 'equip-2', 'equip-3', 'equip-4', 'equip-5',
    'Plenum1_T_C', 'Plenum1_RH_%',
    'Space1_T_C', 'Space1_RH_%',
    'Space2_T_C', 'Space2_RH_%',
    'Space3_T_C', 'Space3_RH_%',
    'Space4_T_C', 'Space4_RH_%',
    'Space5_T_C', 'Space5_RH_%',
    'Plenum_CO2_ppm', 'Plenum_CO2_pred', 'Plenum_CO2_setpoint_ppm', 'Plenum_CO2_internal_gain',
    'Space1_CO2_ppm', 'Space1_CO2_pred', 'Space1_CO2_setpoint_ppm', 'Space1_CO2_internal_gain',
    'Space2_CO2_ppm', 'Space2_CO2_pred', 'Space2_CO2_setpoint_ppm', 'Space2_CO2_internal_gain',
    'Space3_CO2_ppm', 'Space3_CO2_pred', 'Space3_CO2_setpoint_ppm', 'Space3_CO2_internal_gain',
    'Space4_CO2_ppm', 'Space4_CO2_pred', 'Space4_CO2_setpoint_ppm', 'Space4_CO2_internal_gain',
    'Space5_CO2_ppm', 'Space5_CO2_pred', 'Space5_CO2_setpoint_ppm', 'Space5_CO2_internal_gain',
    'doas_fan',
    'fcu_1', 'fcu_2', 'fcu_3', 'fcu_4', 'fcu_5',
    'hex', 'chiller', 'tower', 'boiler',
    'coldw_pump', 'condw_pump', 'hotw_pump',
    'Node2_T_C', 'Node2_Mdot_kg/s', 'Node2_W_Ratio',
    'Node2_SP_T_C', 'Node2_CO2_ppm', 'Node1_T_C',
    'Gas_Facility_E_J', 'Elec_Facility_E_J', 'Elec_HVAC_E_J',
    'CoolingCoils:EnergyTransfer', 'HeatingCoils:EnergyTransfer',
    'ElectricityNet:Facility',
    'General:Cooling:EnergyTransfer', 'Cooling:EnergyTransfer',
]

ELEC_PRICE = 0.11   # €/kWh
GAS_PRICE  = 0.06   # €/kWh
J_TO_KWH   = 1 / 3.6e6

## Utility functions

In [ ]:
# ── Load & preprocess ────────────────────────────────────────────────────────

def load_eplusout(csv_path: str) -> pd.DataFrame:
    """Load an EnergyPlus CSV, fix the 24:00 timestamp, rename columns."""
    df = pd.read_csv(csv_path)
    df['Date/Time'] = df['Date/Time'].str.strip()

    mask_24 = df['Date/Time'].str.contains('24:00:00')
    df.loc[mask_24, 'Date/Time'] = (
        df.loc[mask_24, 'Date/Time'].str.replace('24:00:00', '00:00:00')
    )
    df['Date/Time'] = '2024/' + df['Date/Time']
    df['Date/Time'] = pd.to_datetime(df['Date/Time'], format='%Y/%m/%d %H:%M:%S')
    df.loc[mask_24, 'Date/Time'] += pd.Timedelta(days=1)

    df.columns = COLUMN_NAMES
    return df

In [ ]:
# ── Energy metrics ───────────────────────────────────────────────────────────

def compute_annual_energy(df: pd.DataFrame) -> pd.Series:
    """Return annual energy totals in kWh."""
    cols = ['Gas_Facility_E_J', 'Elec_Facility_E_J', 'Elec_HVAC_E_J']
    return df[cols].sum() * J_TO_KWH


def compute_monthly_energy(df: pd.DataFrame) -> pd.DataFrame:
    """Monthly energy consumption and cost."""
    df = df.copy()
    df['Time'] = pd.to_datetime(df['Time'])
    monthly = df.groupby(df['Time'].dt.to_period('M')).agg(
        elec_fac_kWh=('Elec_Facility_E_J', lambda x: x.sum() * J_TO_KWH),
        elec_hvac_kWh=('Elec_HVAC_E_J', lambda x: x.sum() * J_TO_KWH),
        gas_kWh=('Gas_Facility_E_J', lambda x: x.sum() * J_TO_KWH),
    )
    monthly['elec_fac_cost_eur'] = monthly['elec_fac_kWh'] * ELEC_PRICE
    monthly['elec_hvac_cost_eur'] = monthly['elec_hvac_kWh'] * ELEC_PRICE
    monthly['gas_cost_eur'] = monthly['gas_kWh'] * GAS_PRICE
    monthly['total_cost_eur'] = monthly['elec_fac_cost_eur'] + monthly['gas_cost_eur']
    return monthly


def compute_annual_cost(df: pd.DataFrame) -> pd.Series:
    """Single-row annual cost summary."""
    m = compute_monthly_energy(df)
    return m[['elec_fac_cost_eur', 'elec_hvac_cost_eur', 'gas_cost_eur', 'total_cost_eur']].sum()


def compute_hvac_enduse_energy(df: pd.DataFrame) -> pd.Series:
    """HVAC end-use breakdown in kWh."""
    cols = ['doas_fan', 'fcu_1', 'fcu_2', 'fcu_3', 'fcu_4', 'fcu_5']
    return df[cols].sum() * J_TO_KWH

In [ ]:
# ── CO2 metrics ──────────────────────────────────────────────────────────────

CO2_THRESHOLDS = [770, 970, 1220]
ZONE_CO2_COLS  = [f'Space{i}_CO2_ppm' for i in range(1, 6)]
CLASS_THRESHOLD = 0.90  # 90% of hours must be below limit for classification


def compute_co2_mean(df: pd.DataFrame) -> pd.Series:
    """Mean CO2 per space."""
    return df[ZONE_CO2_COLS].mean()


def compute_co2_exceedances(df: pd.DataFrame) -> pd.DataFrame:
    """Hours above each CO2 threshold per space."""
    return pd.concat(
        {f'>{t} ppm': (df[ZONE_CO2_COLS] > t).sum() for t in CO2_THRESHOLDS},
        axis=1,
    )


def co2_class_90pct(co2_series: pd.Series) -> str:
    """Classify a zone's CO2 based on the 90% rule.

    The zone achieves class X if ≥90% of hours are below thresholds:
      S1: 90% of hours < 770 ppm
      S2: 90% of hours < 970 ppm
      S3: 90% of hours < 1220 ppm
    """
    total = len(co2_series)
    if total == 0:
        return 'No data'
    pct_below_770  = (co2_series <= 770).sum()  / total
    pct_below_970  = (co2_series <= 970).sum()  / total
    pct_below_1220 = (co2_series <= 1220).sum() / total

    if pct_below_770 >= CLASS_THRESHOLD:
        return 'S1'
    if pct_below_970 >= CLASS_THRESHOLD:
        return 'S2'
    if pct_below_1220 >= CLASS_THRESHOLD:
        return 'S3'
    return 'Out of class'


def compute_co2_classification(df: pd.DataFrame) -> pd.DataFrame:
    """Per-space CO2 classification based on 90% compliance rule."""
    rows = []
    total = len(df)
    for col in ZONE_CO2_COLS:
        co2 = df[col]
        rows.append({
            'space': col,
            'max_ppm': co2.max(),
            'mean_ppm': co2.mean(),
            'pct_below_770': 100 * (co2 <= 770).sum() / total if total else 0,
            'pct_below_970': 100 * (co2 <= 970).sum() / total if total else 0,
            'pct_below_1220': 100 * (co2 <= 1220).sum() / total if total else 0,
            'class': co2_class_90pct(co2),
        })
    return pd.DataFrame(rows).set_index('space')

In [ ]:
# ── Temperature metrics ──────────────────────────────────────────────────────

def compute_temperature_limits(df, t_out_col='Outdoor_Tdb_C'):
    """S1 / S2 / S3 temperature limits based on 24-h rolling outdoor mean."""
    t_mean = df[t_out_col].rolling(24, min_periods=24).mean()
    t = pd.to_numeric(t_mean, errors='coerce').to_numpy(dtype=float)

    lower_S1 = np.where(t <= 0, 20.5, np.where(t <= 20, 20.5 + 0.075 * t, 22.0))
    upper_S1 = np.where(t <= 0, 22.0, np.where(t <= 15, 22.5 + 0.166 * t, 25.0))
    lower_S2 = np.where(t <= 0, 20.5, np.where(t <= 20, 20.5 + 0.025 * t, 21.0))
    upper_S2 = np.where(t <= 0, 23.0, np.where(t <= 15, 23.0 + 0.20  * t, 26.0))
    lower_S3 = np.full_like(t, 20.0)
    upper_S3 = np.where(t <= 10, 25.0, 27.0)

    return lower_S1, upper_S1, lower_S2, upper_S2, lower_S3, upper_S3


def classify_temperature(t_in, lower_S1, upper_S1, lower_S2, upper_S2, lower_S3, upper_S3):
    cond_S1 = (t_in >= lower_S1) & (t_in <= upper_S1)
    cond_S2 = (t_in >= lower_S2) & (t_in <= upper_S2)
    cond_S3 = (t_in >= lower_S3) & (t_in <= upper_S3)
    return np.select([cond_S1, cond_S2, cond_S3], ['S1', 'S2', 'S3'], default='Out of class')


def analyze_temperature(df: pd.DataFrame) -> pd.DataFrame:
    """Per-space temperature class breakdown."""
    limits = compute_temperature_limits(df)
    lower_S1, upper_S1, lower_S2, upper_S2, lower_S3, upper_S3 = limits

    rows = []
    for i in range(1, 6):
        col = f'Space{i}_T_C'
        t_in = pd.to_numeric(df[col], errors='coerce')
        classes = classify_temperature(t_in, *limits)
        counts = pd.Series(classes).value_counts()

        s1 = counts.get('S1', 0)
        s2 = counts.get('S2', 0)
        s3 = counts.get('S3', 0)
        out = counts.get('Out of class', 0)
        total = s1 + s2 + s3 + out

        rows.append({
            'space': col,
            'avg_T_C': t_in.mean(),
            'hours_below_20C': (t_in < 20.0).sum(),
            'S1_h': s1, 'S2_h': s2, 'S3_h': s3, 'out_h': out,
            'S1_%': 100 * s1 / total if total else 0,
            'S2_cum_%': 100 * (s1 + s2) / total if total else 0,
            'S3_cum_%': 100 * (s1 + s2 + s3) / total if total else 0,
        })
    return pd.DataFrame(rows).set_index('space')

In [ ]:
# ── Penalty cost functions ────────────────────────────────────────────────────

# CO2 penalty prices (€/h per space, for hours exceeding thresholds)
CO2_P1 = 2    # €/h — above S1 (770 ppm) but ≤ S2 (970 ppm)
CO2_P2 = 10   # €/h — above S2 (970 ppm) but ≤ S3 (1220 ppm)
CO2_P3 = 50   # €/h — above S3 (1220 ppm)

# Temperature penalty prices (€/h per space, for hours outside class band)
TEMP_P1 = 1    # €/h — outside S1 but inside S2
TEMP_P2 = 5    # €/h — outside S2 but inside S3
TEMP_P3 = 25   # €/h — outside S3


def compute_co2_penalty(df: pd.DataFrame) -> pd.DataFrame:
    """
    CO2 penalty cost per space (hourly).
    C_CO2 = p1 * h_{S1→S2} + p2 * h_{S2→S3} + p3 * h_{>S3}
    """
    rows = []
    for col in ZONE_CO2_COLS:
        co2 = df[col]
        h_s1_s2 = ((co2 > 770) & (co2 <= 970)).sum()
        h_s2_s3 = ((co2 > 970) & (co2 <= 1220)).sum()
        h_above  = (co2 > 1220).sum()
        cost = CO2_P1 * h_s1_s2 + CO2_P2 * h_s2_s3 + CO2_P3 * h_above
        rows.append({
            'space': col.replace('_CO2_ppm', ''),
            'h_above_S1': h_s1_s2,
            'h_above_S2': h_s2_s3,
            'h_above_S3': h_above,
            'penalty_eur': cost,
        })
    return pd.DataFrame(rows).set_index('space')


def compute_temperature_penalty(df: pd.DataFrame) -> pd.DataFrame:
    """
    Temperature penalty per space — hourly cost for time outside class bands.

    Each hour a zone spends outside a class band adds a penalty:
      outside S1 but in S2 → TEMP_P1 €/h
      outside S2 but in S3 → TEMP_P2 €/h
      outside S3           → TEMP_P3 €/h

    The achieved classification still uses the 90% compliance rule.
    """
    limits = compute_temperature_limits(df)
    lower_S1, upper_S1, lower_S2, upper_S2, lower_S3, upper_S3 = limits

    rows = []
    for i in range(1, 6):
        col = f'Space{i}_T_C'
        t_in = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
        total = len(t_in)

        in_s1 = (t_in >= lower_S1) & (t_in <= upper_S1)
        in_s2 = (t_in >= lower_S2) & (t_in <= upper_S2)
        in_s3 = (t_in >= lower_S3) & (t_in <= upper_S3)

        # Hours in each violation tier
        h_s1_only = (~in_s1 & in_s2).sum()   # outside S1, inside S2
        h_s2_only = (~in_s2 & in_s3).sum()   # outside S2, inside S3
        h_out     = (~in_s3).sum()            # outside S3

        penalty = TEMP_P1 * h_s1_only + TEMP_P2 * h_s2_only + TEMP_P3 * h_out

        # 90% classification
        pct_s1 = 100 * in_s1.sum() / total if total else 0
        pct_s2 = 100 * in_s2.sum() / total if total else 0
        pct_s3 = 100 * in_s3.sum() / total if total else 0
        pct_req = CLASS_THRESHOLD * 100
        if pct_s1 >= pct_req:
            achieved = 'S1'
        elif pct_s2 >= pct_req:
            achieved = 'S2'
        elif pct_s3 >= pct_req:
            achieved = 'S3'
        else:
            achieved = 'Out of class'

        rows.append({
            'space': f'Space{i}',
            'achieved_class': achieved,
            'h_outside_S1': h_s1_only,
            'h_outside_S2': h_s2_only,
            'h_outside_S3': h_out,
            'S1_%': pct_s1,
            'penalty_eur': penalty,
        })
    return pd.DataFrame(rows).set_index('space')


def compute_total_cost(df: pd.DataFrame) -> dict:
    """Energy cost + CO2 penalty + temperature penalty."""
    monthly = compute_monthly_energy(df)
    energy_cost = monthly['total_cost_eur'].sum()
    co2_cost = compute_co2_penalty(df)['penalty_eur'].sum()
    temp_cost = compute_temperature_penalty(df)['penalty_eur'].sum()
    return {
        'energy_cost_eur': energy_cost,
        'co2_penalty_eur': co2_cost,
        'temp_penalty_eur': temp_cost,
        'total_cost_eur': energy_cost + co2_cost + temp_cost,
    }

## Load models

Add or remove entries in the `MODELS` dict to compare different runs.

In [ ]:
# ── Define models to compare ────────────────────────────────────────────────
# key  = display name
# value = path to the eplusout.csv

MODELS = {
    'RBC Full-On':   'rbc_full_on/eplus_out/eplusout.csv',
    'RBC Scheduled': 'rbc_scheduled/eplus_out/eplusout.csv',
    'DRL':           'drl/drl_output/run_1/eplusout.csv',
}

# Load all models
models = {name: load_eplusout(path) for name, path in MODELS.items()}
print(f'Loaded {len(models)} model(s): {list(models.keys())}')

## Energy comparison

In [ ]:
# ── Annual energy ────────────────────────────────────────────────────────────

annual_energy = pd.DataFrame({name: compute_annual_energy(df) for name, df in models.items()})
print('Annual energy use (kWh):')
print(annual_energy.round(2))
print()

# Savings relative to the first model
ref_name = list(models.keys())[0]
for name in list(models.keys())[1:]:
    diff = annual_energy[ref_name] - annual_energy[name]
    pct  = diff / annual_energy[ref_name] * 100
    print(f'Savings {name} vs {ref_name}:')
    print(pd.DataFrame({'kWh': diff, '%': pct}).round(2))
    print()

In [ ]:
# ── Annual cost ──────────────────────────────────────────────────────────────

annual_cost = pd.DataFrame({name: compute_annual_cost(df) for name, df in models.items()})
print('Annual cost (€):')
print(annual_cost.round(2))

In [ ]:
# ── Monthly cost plot ────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 5))
for name, df in models.items():
    m = compute_monthly_energy(df)
    ax.plot(m.index.astype(str), m['total_cost_eur'], marker='o', label=name)

ax.set_ylabel('Monthly energy cost (€)')
ax.set_title('Monthly Energy Cost Comparison')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Annual energy bar chart ──────────────────────────────────────────────────

labels = annual_energy.index
x = np.arange(len(labels))
width = 0.8 / len(models)

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(models):
    ax.bar(x + i * width - 0.4 + width / 2, annual_energy[name], width, label=name)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Annual Energy Use (kWh)')
ax.set_title('Annual Energy Use Comparison')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── HVAC end-use breakdown ───────────────────────────────────────────────────

hvac_enduse = pd.DataFrame({name: compute_hvac_enduse_energy(df) for name, df in models.items()})
print('HVAC end-use breakdown (kWh):')
print(hvac_enduse.round(2))

## CO2 comparison

In [ ]:
# ── Mean CO2 ─────────────────────────────────────────────────────────────────

co2_means = pd.DataFrame({name: compute_co2_mean(df) for name, df in models.items()})
print('Average CO2 concentration by space (ppm):')
print(co2_means.round(2))

In [ ]:
# ── CO2 exceedances ──────────────────────────────────────────────────────────

for name, df in models.items():
    print(f'\nHours above thresholds — {name}:')
    print(compute_co2_exceedances(df))

In [ ]:
# ── CO2 classification ───────────────────────────────────────────────────────

for name, df in models.items():
    print(f'\nCO2 classification — {name}:')
    print(compute_co2_classification(df))

## Temperature comparison

In [ ]:
# ── Temperature class analysis ───────────────────────────────────────────────

for name, df in models.items():
    print(f'\n═══ Temperature analysis — {name} ═══')
    print(analyze_temperature(df).round(2))

In [ ]:
# ── Per-space temperature time-series with class bands ───────────────────────

def plot_temperature_comparison(models_dict, space_i=1):
    """Plot temperature for one space across all models with S1/S2/S3 bands."""
    col = f'Space{space_i}_T_C'
    fig, ax = plt.subplots(figsize=(18, 6))

    # Use limits from first model (same weather file → same limits)
    first_df = list(models_dict.values())[0]
    lower_S1, upper_S1, lower_S2, upper_S2, lower_S3, upper_S3 = compute_temperature_limits(first_df)
    x = pd.to_datetime(first_df['Time'])

    ax.fill_between(x, lower_S3, upper_S3, alpha=0.06, label='S3 band')
    ax.fill_between(x, lower_S2, upper_S2, alpha=0.08, label='S2 band')
    ax.fill_between(x, lower_S1, upper_S1, alpha=0.10, label='S1 band')

    for name, df in models_dict.items():
        ax.plot(pd.to_datetime(df['Time']), df[col], linewidth=0.8, alpha=0.9, label=f'{name}')

    ax.set_title(f'{col} — All Models')
    ax.set_xlabel('Time')
    ax.set_ylabel('Temperature (°C)')
    ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


for i in range(1, 6):
    plot_temperature_comparison(models, space_i=i)

## Total Cost (Energy + Penalties)

In [ ]:
# ── CO2 penalty comparison ────────────────────────────────────────────────────

print(f'CO2 penalties: p1={CO2_P1} €/h (>770), p2={CO2_P2} €/h (>970), p3={CO2_P3} €/h (>1220)')
print()

for name, df in models.items():
    pen = compute_co2_penalty(df)
    print(f'CO2 penalty — {name}:')
    print(pen)
    print(f'Total: {pen["penalty_eur"].sum():.2f} €\n')

In [ ]:
# ── Temperature penalty comparison ────────────────────────────────────────────

print(f'Temperature penalties (hourly): outside S1={TEMP_P1} €/h, outside S2={TEMP_P2} €/h, outside S3={TEMP_P3} €/h')
print()

for name, df in models.items():
    pen = compute_temperature_penalty(df)
    print(f'Temperature penalty — {name}:')
    print(pen)
    print(f'Total: {pen["penalty_eur"].sum():.2f} €\n')

In [ ]:
# ── Total cost comparison table + bar chart ───────────────────────────────────

cost_rows = {}
for name, df in models.items():
    cost_rows[name] = compute_total_cost(df)

cost_df = pd.DataFrame(cost_rows).T
cost_df.index.name = 'model'

print('Total cost breakdown (€):')
print(cost_df.round(2))
print()

# Savings relative to first model
ref_name = cost_df.index[0]
for name in cost_df.index[1:]:
    diff = cost_df.loc[ref_name, 'total_cost_eur'] - cost_df.loc[name, 'total_cost_eur']
    pct  = diff / cost_df.loc[ref_name, 'total_cost_eur'] * 100
    print(f'{name} vs {ref_name}: {diff:+.2f} € ({pct:+.1f} %)')

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(cost_df))
w = 0.5

ax.bar(x, cost_df['energy_cost_eur'], w, label='Energy cost')
ax.bar(x, cost_df['co2_penalty_eur'], w, bottom=cost_df['energy_cost_eur'], label='CO2 penalty')
ax.bar(x, cost_df['temp_penalty_eur'], w,
       bottom=cost_df['energy_cost_eur'] + cost_df['co2_penalty_eur'], label='Temp. penalty')

# Total label on top of each bar
for i, name in enumerate(cost_df.index):
    total = cost_df.loc[name, 'total_cost_eur']
    ax.text(i, total + total * 0.01, f'{total:.0f} €', ha='center', va='bottom', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(cost_df.index)
ax.set_ylabel('Cost (€)')
ax.set_title('Total Cost Comparison (Energy + CO2 + Temperature Penalties)')
ax.legend()
plt.tight_layout()
plt.show()